![Platform+](images/banner.png)

# Lab 3: Understand Access / RBAC
_Hands-on Snowflake for data practitioners_

## Why can't I see this table? Grant access the right way.

> **Persona track: Data Engineer** — same lab, your angle.

:point_right: **In this lesson, you'll reproduce an access-denied error, understand why it happens, fix it by granting the right privilege, and learn techniques to manage access across multiple roles.**

The scenario: Your coffee roastery has an ORDERS table and a FEEDBACK table. A new analyst role has been set up — but someone forgot to grant access to the feedback data. You'll experience the failure, then fix it.

We'll cover:
1. How access flows in Snowflake: **privilege → role → user**
2. Setting up the lab scenario (tables + partial grants)
3. Reproducing an access-denied error (the #1 new-user question)
4. Granting the missing privilege to fix it
5. Automating access with future grants
6. Schema-wide grants vs per-object grants
7. The engineer's deployment pattern: table + grants together

:books: **By the end you'll be able to:**
- Explain the full grant chain: warehouse → database → schema → table
- Understand why switching roles can make tables "disappear"
- Grant and revoke SELECT on a table (as the object owner)
- Use future grants to automate access for new objects
- Choose between schema-wide and per-object grant patterns
- Deploy tables with grants as a single atomic workflow
- Use CoCo as an access automation partner
- Apply the principle of least privilege

**Time:** ~35 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every Snowflake session operates under a specific role and warehouse. The role determines what you can see and do — the warehouse provides the compute.

> :bulb: **Well-Architected principle:** Never use ACCOUNTADMIN for daily work or object creation. Use the least-privileged role that can accomplish the task.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};


#### Step 1: Create the tables in your database

In the lab setup below you will copy two tables from the shared `PLATPLUS_DB` into your personal database. We do this so **you own the tables** — which means you can grant privileges on them without needing SECURITYADMIN.

**Why this matters:**
- In Snowflake, the **owner** of an object (the role that created it) can grant privileges on it
- By creating these tables under `{{my_role}}`, you become the owner and can control who else accesses them
- This simulates a real scenario where a team owns their data and manages access to it

| Table | What it contains | Who should have access |
|-------|-----------------|------------------------|
| `ORDERS` | Customer order history | Both {{my_role}} and {{my_analyst_role}} |
| `FEEDBACK` | Customer feedback/reviews | {{my_role}} only (initially) — we'll "forget" to grant {{my_analyst_role}} |


#### Step 2: Grant partial access (deliberately incomplete)

The cell below grants `{{my_analyst_role}}` access to the database, schema, and the ORDERS table — but **intentionally omits** the FEEDBACK table. This sets up the access-denied scenario you'll troubleshoot in Section 3.

**Why we grant in layers (warehouse → database → schema → table):**

Snowflake access is hierarchical. You need grants at **every level** to run a query:

```
Warehouse (USAGE)  →  Database (USAGE)  →  Schema (USAGE)  →  Table (SELECT)
```

Missing any link in the chain = failure. No warehouse = can't execute. No database = can't reference objects. No schema = can't see tables. No SELECT = access denied.

---

**Best practice:** When onboarding a new role, always grant the full chain. Use a checklist:
1. USAGE on warehouse (provides compute — without this, nothing runs)
2. USAGE on database
3. USAGE on schema
4. Specific privilege on object (SELECT, INSERT, etc.)

> :bulb: In this lab, `{{my_analyst_role}}` already has USAGE on `PLATPLUS_WH` (granted during environment setup by the platform admin). That's why the cell below only grants the database → schema → table chain.

> :wrench: **Real-world note — two paths to compute:**
>
> | Path | When to use | Example |
> |------|-------------|--------|
> | **Use a shared warehouse** | Most common for analysts. A platform admin grants USAGE on an existing warehouse. | `GRANT USAGE ON WAREHOUSE shared_wh TO ROLE analyst_rl;` |
> | **Create your own warehouse** | When you have `CREATE WAREHOUSE` privilege (common for data engineers). You own it and can grant access to others. | `CREATE WAREHOUSE my_team_wh WAREHOUSE_SIZE='XSMALL'; GRANT USAGE ON WAREHOUSE my_team_wh TO ROLE analyst_rl;` |
>
> In this lab we use the shared `PLATPLUS_WH`. In production, data engineers often create dedicated warehouses sized for their workloads (e.g., a small warehouse for ad-hoc queries, a larger one for heavy pipeline loads).

---

**Use cases for partial access:**
| Scenario | What you grant | What you withhold |
|----------|---------------|-------------------|
| Analyst sees orders but not PII | SELECT on orders | No access to customer_details table |
| Junior dev reads but can't write | SELECT on all tables | No INSERT, UPDATE, DELETE |
| External partner sees summary views | SELECT on views only | No access to base tables |


In [ ]:
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RBAC_LAB;

## 1 · How access works in Snowflake

Access in Snowflake follows a simple chain:

```
PRIVILEGE  →  ROLE  →  USER
(what)        (who)    (person)
```

- **Privileges** are permissions on objects (SELECT on a table, USAGE on a schema)
- **Roles** collect privileges — they define "what this job function can do"
- **Users** are assigned roles — they inherit whatever privileges those roles hold

**The key rule:** In Role-Based Access Control (RBAC), you grant privileges to **roles**. Users then inherit whatever privileges their assigned roles hold.

**Why?** Because people change roles, leave companies, get promoted. If you grant to users, you have to individually manage every person. If you grant to roles, you manage the role once and assign people to it.

> :bulb: **Real-world context:** "Why can't I see this table?" is the #1 support question from new Snowflake users. The answer is almost always: your current role doesn't have a grant on that object. The data didn't disappear — your role just can't see it.

**The system-defined role hierarchy:**
```
ACCOUNTADMIN
  ├── SECURITYADMIN  (manages grants + roles)
  │     └── USERADMIN  (manages users)
  └── SYSADMIN  (manages warehouses + databases)
        └── custom roles (yours!)
              └── PUBLIC  (everyone)
```

Higher roles inherit privileges from lower roles. A custom role should always be granted to SYSADMIN (or another parent) so admins can manage objects created by that role.

## 2 · Set up the lab scenario

As the owner of your database (`{{my_role}}`), you'll create two tables and grant **partial** access to a second role (`{{my_analyst_role}}`). This simulates a real scenario: an analyst is set up with access to orders data, but someone forgot to include the feedback table.

> :bulb: **Where does `{{my_analyst_role}}` come from?** This role was pre-created by your instructor as part of the lab environment setup. It has warehouse usage (so it can run queries) but **no access to any database objects** by default. This is intentional — it represents a new team member's role before anyone has granted them access to specific data. Your job in this lab is to grant (and troubleshoot) that access.

> :point_right: **A note on Sections 1–4:** The RBAC fundamentals (grant chains, access-denied errors, fixing missing privileges) are universal concepts that apply to all personas. Sections 1–4 cover this shared foundation. **Sections 5–7 are where the Data Engineer-specific patterns diverge** — future grants, schema-wide automation, and deployment workflows that are uniquely part of your role.


In [ ]:
-- Copy two tables into your personal database for the lab
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.ORDERS   AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.FEEDBACK AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK;

In [ ]:
-- Warehouse USAGE on PLATPLUS_WH was granted to {{my_analyst_role}} during environment setup.
-- Here we grant the remaining links in the chain: database → schema → table
GRANT USAGE  ON DATABASE {{my_db}}                 TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB        TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS TO ROLE {{my_analyst_role}};
-- Note: NO grant on FEEDBACK — that's intentional


> :bulb: **What just happened — decoded:**
>
> | SQL Statement | What it means |
> |---|---|
> | `GRANT USAGE ON WAREHOUSE` | Allows the role to use compute (granted during setup on `PLATPLUS_WH`) |
> | `GRANT USAGE ON DATABASE` | Allows the role to "see" the database exists and enter it |
> | `GRANT USAGE ON SCHEMA` | Allows the role to see the schema and list objects inside |
> | `GRANT SELECT ON TABLE` | Allows the role to query (read) data from that specific table |
>
> **You need ALL FOUR** to query a table. Missing any one link in the chain = failure. The warehouse grant was already in place (from environment setup), so we only needed to add the database → schema → table grants here.

## 3 · Switch roles and hit the wall

Now become `{{my_analyst_role}}` — the limited role. You'll run two queries: one that works and one that fails. The data is the same — the only difference is whether the role has a grant.

**Why this matters (data engineer perspective):**
- **You build the pipelines that create tables** — If downstream consumers (analysts, dashboards, apps) can't access what you produced, your pipeline's output is invisible. Understanding access errors helps you proactively prevent them.
- **You set up roles and grants as part of deployment** — In production, creating a table isn't done until access is granted. A table nobody can read is a table that doesn't exist for business purposes.
- **Debugging access issues is faster than filing tickets** — When a colleague says "I can't see the table you just created," you can diagnose the problem in seconds if you understand the grant chain.


In [ ]:
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;

-- This works: {{my_analyst_role}} has SELECT on ORDERS
SELECT COUNT(*) AS order_count FROM {{my_db}}.RBAC_LAB.ORDERS;


> :white_check_mark: **Why this worked:** In Section 2, you granted `{{my_analyst_role}}` the full access chain — `USAGE` on the database, `USAGE` on the schema, and `SELECT` on ORDERS. All three links are present, so the query succeeds.


> :stop_sign: **The next cell will intentionally fail.** This is expected — it demonstrates what happens when a role is missing a privilege. When you see the error and a **"Fix Error" button appears, do NOT click it.** We want you to understand the error first and fix it yourself in the next section.

In [ ]:
-- This FAILS: {{my_analyst_role}} has NO grant on FEEDBACK
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Read the error message carefully.** You should see something like:
>
> `Object 'FEEDBACK' does not exist or not authorized. Your primary role {{my_analyst_role}} must have at least one privilege granted on TABLE FEEDBACK.`
>
> The table EXISTS. The data is THERE. Your role simply doesn't have a GRANT to read it. This is RBAC working exactly as designed — protecting data from unauthorized access.
>
> **The distinction:** "Does not exist or not authorized" does NOT necessarily mean the table is missing — it may well exist, but your current role can't see it. Snowflake deliberately uses this vague message to avoid leaking information about what objects exist to unauthorized roles.


## 4 · Grant the missing privilege — as the owner

You don't need SECURITYADMIN to fix this. You **own** these tables (you created them with `{{my_role}}`), so you can grant privileges on them.

**Why this matters (data engineer perspective):**
- **You create most of the objects in your schema** — Tables, views, stages, pipes — as the creator, you're the owner by default. That means granting access is part of your delivery workflow, not someone else's problem.
- **Deployment isn't complete until access is granted** — A pipeline that loads data but doesn't grant READ to consumers is only half done. Treat GRANTs as part of your DDL scripts.
- **Least privilege protects your pipeline** — Granting only SELECT (not INSERT, UPDATE, DELETE) means consumers can read your output without accidentally corrupting it.

---

**Use cases where data engineers grant access:**
| Scenario | What you grant | To whom |
|----------|---------------|--------|
| New table for analysts | SELECT on the table | Analytics role |
| Staging tables for ETL | SELECT + INSERT on staging | Pipeline service role |
| Shared reference data | SELECT on views only | Multiple consumer roles |
| Temporary project access | SELECT (then REVOKE when project ends) | Project-specific role |

---

**Best practice — Least privilege:** Grant only the specific privilege needed (SELECT), on the specific object needed (FEEDBACK), to the specific role that needs it ({{my_analyst_role}}). Never grant more than required.

> :bulb: **Well-Architected principle:** In production, GRANTs should be version-controlled alongside your DDL. When you create a table in a deployment script, include the GRANT statement in the same script — never rely on manual grants after the fact.


In [ ]:
-- Switch back to {{my_role}} (the owner) to grant the missing privilege
USE ROLE {{my_role}};

-- {{my_analyst_role}} already has: WAREHOUSE (USAGE) + DATABASE (USAGE) + SCHEMA (USAGE)
-- The only missing link is SELECT on FEEDBACK
GRANT SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK TO ROLE {{my_analyst_role}};


In [ ]:
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Same query, same data — the only thing that changed was a GRANT to the role.** That's RBAC in action.
>
> The pattern: 1) User hits access-denied → 2) Identify missing privilege → 3) Grant it to the role → 4) User retries — works.

---
## :robot_face: CoCo Assist

As a data engineer, CoCo isn't just for writing pipelines — it's your **access automation partner**. Instead of manually writing GRANT statements for every new table, you can ask CoCo to generate complete access setups in seconds. Here are three examples of how you'd use it in the real world.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — don't just glance at it. The explanation is where the learning happens.
> 2. **CoCo is interactive** — if something in the response doesn't make sense, ask a follow-up question. "What do you mean by 'inherited privilege'?" or "Can you show me an example?" The conversation continues until *you* understand.
> 3. **The lab effort is on you** — the prompts below are starting points. Your real learning comes from using the CoCo interface, reading the responses, and asking your own questions based on what you see.
> 4. **There's no single right answer** — CoCo may explain things differently each time. That's fine. Focus on understanding the concept, not memorizing the exact words.

---

**Example 1: Understand RBAC fundamentals**

A colleague asks: *"What's the difference between granting a privilege to a role vs directly to a user?"*

| Term | What it means |
|---|---|
| `GRANT ... TO ROLE` | Assigns a privilege to a role — anyone with that role inherits it |
| `GRANT ... TO USER` | Assigns directly to one person (UBAC) — doesn't scale |
| RBAC (Role-Based) | Standard approach — grant to roles, assign users to roles |
| UBAC (User-Based) | Direct-to-user — useful for private dev, not recommended for production |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Explain the difference between granting a privilege to a role vs a user in Snowflake. When should I use each approach?</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Give me a real-world scenario where granting directly to a user would be appropriate.</pre>

---

**Example 2: Check your own privileges**

You want to know what access your current role has on the tables you just queried.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What privileges does {{my_analyst_role}} have on the ORDERS and FEEDBACK tables in {{my_db}}.RBAC_LAB? Show me the grants.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Why does {{my_analyst_role}} have SELECT but not INSERT? What would happen if I tried to INSERT?</pre>

This is the self-diagnosis skill — instead of guessing why a query failed, you can verify exactly what grants exist (or are missing).

---

**Example 3: Investigate past failures in query history**

Query history logs every query that ran — including failures. To troubleshoot something that *just* failed, use the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function — it has no reporting latency, so a failure from moments ago shows up immediately (its `EXECUTION_STATUS` for a failed query is `FAILED_WITH_ERROR`). You can ask CoCo to find recent access-denied errors and explain why they happened.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Using the real-time INFORMATION_SCHEMA.QUERY_HISTORY table function, check for any queries against the FEEDBACK table in {{my_db}}.RBAC_LAB that failed with an error (EXECUTION_STATUS = 'FAILED_WITH_ERROR') in the last hour. Show me the error messages and explain why they failed.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What grant would have prevented that failure?</pre>

Query history records the SQL text, user, role, status, and error message for every query executed in the account. Two sources exist: the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function used above (best for troubleshooting something that just happened), and the `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` view (a 365-day history with up to ~2 hours of latency — better for audits and access reviews). It's invaluable for troubleshooting — you can see exactly what was attempted, by whom, and what went wrong.

> :bulb: **Two things to know about the table function.** It only reaches back **7 days** — query a wider window and it returns an error, so use the `ACCOUNT_USAGE` view for older history. And it returns **only your own queries** unless your role holds `MONITOR`/`OPERATE` on the warehouse (or is `ACCOUNTADMIN`) — a reminder that even query visibility follows RBAC. Here, the failed query was yours, so it shows up regardless of which role is active.

---

> :bulb: **These are just examples.** CoCo can help with any access-related question: "Why can't ROLE_X see this view?", "What's the minimum set of grants for a dashboard to read my table?", "Show me all roles that have access to the SALES schema." The point is that access troubleshooting doesn't have to mean memorizing SHOW GRANTS syntax — you can ask in plain English and follow up until it clicks.


---
## 5 · Your angle — Data Engineer: Grant at scale

Granting SELECT on every individual table is tedious. As an engineer who creates new tables regularly, you need downstream consumers to **automatically** get access.

**Solution: Future grants.** Tell Snowflake: "Whenever a new table is created in this schema, automatically grant SELECT to this role."

**Why this matters (data engineer perspective):**
- **You create tables constantly** — Every new pipeline, every new staging table, every new materialized view. If each one requires a manual GRANT ticket, you're creating an operational bottleneck.
- **Consumers shouldn't wait for access** — When your pipeline creates a new daily summary table, analysts should be able to query it immediately — not hours later after an admin processes a ticket.
- **Future grants are part of infrastructure-as-code** — They're set once per schema and apply forever. This is the "set it and forget it" pattern for access management.

> :point_right: **How this differs from the Platform Admin's perspective:** You set up future grants as part of your deployment — ensuring your pipeline's output is immediately usable. The Platform Admin *reviews and governs* those grants — ensuring they're not too broad, revoking them if the access model changes, and auditing who has access to what. **You're the creator; they're the auditor.** Both roles touch the same SQL, but for different reasons.

**Use cases:**
| Scenario | Future grant needed | Why |
|----------|-------------------|-----|
| Daily/hourly pipeline creates new tables | `ON FUTURE TABLES IN SCHEMA` | New outputs are immediately queryable |
| Dynamic table creation (date-partitioned) | `ON FUTURE TABLES IN SCHEMA` | Each new partition table auto-grants |
| New schemas for new projects | `ON FUTURE SCHEMAS IN DATABASE` | Project schemas get standard access |
| Views built on top of raw tables | `ON FUTURE VIEWS IN SCHEMA` | Consumers access views, not base tables |

**Best practice:** Always pair `ON ALL TABLES` (existing) with `ON FUTURE TABLES` (new). Without both, you either miss current tables or future ones.

In [ ]:
-- Apply future grants using the training helper procedure
-- (In production, you'd run the GRANT statements directly with MANAGE GRANTS privilege)
USE ROLE {{my_role}};
CALL PLATPLUS_DB.PUBLIC.APPLY_FUTURE_GRANTS('{{my_db}}.RBAC_LAB', '{{my_analyst_role}}');


> :bulb: **What just happened — in detail:**
>
> You called a stored procedure that executed these two statements **on your behalf** using ACCOUNTADMIN privileges:
>
> ```sql
> -- What the procedure ran internally (as ACCOUNTADMIN):
> GRANT SELECT ON ALL TABLES IN SCHEMA {{my_db}}.RBAC_LAB TO ROLE {{my_analyst_role}};
> GRANT SELECT ON FUTURE TABLES IN SCHEMA {{my_db}}.RBAC_LAB TO ROLE {{my_analyst_role}};
> ```
>
> | Statement | Effect |
> |---|---|
> | `ON ALL TABLES` | Grants SELECT on every table that exists RIGHT NOW |
> | `ON FUTURE TABLES` | Grants SELECT on every table CREATED LATER in this schema |
>
> You need BOTH for complete coverage — without `ON ALL`, existing tables are missed. Without `ON FUTURE`, new tables require manual grants.

> :warning: **Why a procedure instead of running GRANT directly?**
>
> The `GRANT SELECT ON FUTURE TABLES` statement requires [`MANAGE GRANTS`](https://community.snowflake.com/s/article/What-is-MANAGE-GRANTS-privilege) — an account-level privilege that allows granting/revoking **anything on any object** in the entire account. That's far too powerful for a lab environment (or for most production roles).
>
> Instead, your instructor created a stored procedure with `EXECUTE AS OWNER` — meaning it runs with its creator's privileges (ACCOUNTADMIN), not yours. You get the result without needing the dangerous privilege.
>
> **This is actually a best practice in production:**
>
> | Pattern | What it solves |
> |---------|---------------|
> | Wrap elevated operations in `EXECUTE AS OWNER` procedures | Teams get self-service without dangerous privileges |
> | Procedure validates inputs before executing | Prevents accidental over-granting (e.g., granting to wrong role) |
> | Procedure is auditable | You can see who called it and when via query history |
> | Procedure is grantable | Only specific roles can call it — fine-grained access control |
>
> In production, your platform admin would create procedures like this for common operations (apply future grants, onboard a new consumer role, revoke project access) and grant USAGE to your engineering role. You get self-service; they maintain governance.

> :point_right: **If you had MANAGE GRANTS (or were ACCOUNTADMIN), you'd run this directly:**
> ```sql
> USE ROLE ACCOUNTADMIN;  -- or a role with MANAGE GRANTS
> GRANT SELECT ON ALL TABLES IN SCHEMA my_db.RBAC_LAB TO ROLE {{my_analyst_role}};
> GRANT SELECT ON FUTURE TABLES IN SCHEMA my_db.RBAC_LAB TO ROLE {{my_analyst_role}};
> ```
> The procedure wraps exactly this — nothing more, nothing less.


### Test it: Create a new table and verify auto-access

Future grants are set — but do they actually work? Let's prove it.

**What we're testing:** If you create a brand new table right now (as `{{my_role}}`), can `{{my_analyst_role}}` query it immediately — without running any additional GRANT statement?

**What we're proving:** Future grants eliminate the manual step. The moment a table exists, the consumer role already has SELECT on it. No ticket, no waiting, no second command.

**The steps:**
1. Create a new summary table (as {{my_role}} — the owner/engineer)
2. Switch to {{my_analyst_role}} (the consumer)
3. Query the new table — if it succeeds, future grants are working

If this works, it means every table your pipeline creates from now on in this schema is automatically accessible to {{my_analyst_role}}.


In [ ]:
USE ROLE {{my_role}};
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.DAILY_SALES AS
  SELECT ORDER_TS::DATE AS order_date, COUNT(*) AS num_orders, SUM(TOTAL_AMOUNT) AS revenue
  FROM {{my_db}}.RBAC_LAB.ORDERS GROUP BY ORDER_TS::DATE;


In [ ]:
-- Can {{my_analyst_role}} see it WITHOUT a new grant?
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.DAILY_SALES LIMIT 5;


> :mag_right: **It works!** No additional GRANT needed. Future grants handle it automatically.
>
> **Engineer takeaway:** Set up future grants once per schema — downstream consumers never wait for access tickets again.

---
### :robot_face: CoCo: Generate a complete access setup

Now that you understand future grants, use CoCo to generate what a production-ready access configuration looks like.

**Jargon decoder:**
| Term | What it means |
|---|---|
| `ON ALL TABLES` | Every existing table in the schema |
| `ON FUTURE TABLES` | Every table created in this schema from now on |
| `ON ALL SCHEMAS` / `ON FUTURE SCHEMAS` | Same pattern but at the database level |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write GRANT statements that would give {{my_analyst_role}} full read-only access to all current AND future tables and views in a schema called RBAC_LAB. Include USAGE grants on the warehouse, database, and schema. Show me both the ON ALL and ON FUTURE statements.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What happens if I only do ON FUTURE but forget ON ALL? Will existing tables be accessible?</pre>

> :warning: **Note:** CoCo will generate the GRANT statements directly. In this lab you **can't run them yourself** (requires MANAGE GRANTS), but you've already applied them via the helper procedure above. The value here is understanding what the full access setup looks like — this is what you'd put in a deployment script or request from your platform admin.


---
## 6 · Extend: When to use per-object vs schema-wide grants

As a data engineer, you design the schema layout — and how you organize schemas directly impacts how access is managed.

**Why this matters (data engineer perspective):**
- **Schema design IS access design** — If sensitive tables live alongside public tables, you're forced into tedious per-object grants. Separating them into different schemas lets you use clean schema-wide grants.
- **Your schema layout decision today creates operational cost forever** — A poorly designed schema means manual grant tickets for every new table. A well-designed one means future grants handle everything automatically.

---

| Approach | Best for | Risk | Data engineer action |
|----------|----------|------|---------------------|
| Per-object (`GRANT ON TABLE x`) | Sensitive/PII data | Tedious at scale | Use sparingly — only for exceptions |
| Schema-wide (`ON ALL/FUTURE TABLES`) | Analytics schemas, shared data | Broader access — new tables auto-shared | Default choice for output schemas |
| Schema separation | Mixed-sensitivity data | More schemas to manage | Put PII in a separate schema with tighter grants |
| Database roles | External data sharing | More complex setup | For cross-account sharing scenarios |

---

**Best practice — Schema-as-access-boundary:**
```
my_db/
├── RAW/          (schema-wide: only pipeline roles)
├── ANALYTICS/    (schema-wide: all analyst roles via future grants)
└── SENSITIVE/    (per-object: PII data, restricted roles only)
```

By organizing data into schemas by sensitivity level, you can use schema-wide grants for 90% of cases and per-object grants only for the remaining 10%. This is the data engineer's key design contribution to access management.

> :bulb: **Well-Architected:** Use schema-wide grants for analytics schemas. Use per-object grants for sensitive data. Separate sensitive data into its own schema — this one design decision eliminates most access management overhead.

---
## 7 · The engineer's deployment pattern: table + grants together

In production, a data engineer never creates a table without also granting access in the same script. This is the fundamental difference between how engineers and analysts think about access — **grants are part of your deployment, not an afterthought.**

> :point_right: **How this relates to Section 5 (future grants):** These two patterns are complementary, not contradictory.
>
> | Scenario | Which pattern to use |
> |----------|---------------------|
> | Schema with future grants already configured | Future grants handle it — no explicit GRANT needed in your script |
> | New schema, new project, or one-off table | Deployment pattern — include the GRANT explicitly |
> | Cross-schema or cross-database access | Deployment pattern — future grants are per-schema only |
> | Someone else's schema (you don't control grants) | Deployment pattern — you must explicitly grant |
>
> **Think of it this way:** Future grants are the *automation* that eliminates the need for explicit grants in established schemas. The deployment pattern is the *universal fallback* that always works regardless of whether future grants are in place. In practice, engineers use both — future grants for established output schemas, explicit grants for new/one-off/cross-schema deployments.

---

**The pattern:**
```sql
-- Always ship these together — never one without the other
CREATE TABLE my_schema.new_table AS ...;
GRANT SELECT ON TABLE my_schema.new_table TO ROLE consumer_role;
```

---

**Why this matters:**
- A table nobody can read is a table that doesn't exist for business purposes
- If you separate creation from granting, there's always a window where consumers can't see the data
- In CI/CD pipelines, the GRANT must be in the same deployment script as the DDL
- Even if future grants exist, including an explicit GRANT is a safe "belt and suspenders" approach — it's idempotent (running it twice doesn't hurt)

The cell below demonstrates this pattern — creating a table and granting in one atomic deployment:

In [ ]:
-- The engineer's deployment pattern: CREATE + GRANT in one script
USE ROLE {{my_role}};

-- Deploy a new summary table (simulates what your pipeline produces)
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.ORDER_SUMMARY AS
  SELECT ORDER_TS::DATE AS order_date, COUNT(*) AS num_orders, SUM(TOTAL_AMOUNT) AS revenue
  FROM {{my_db}}.RBAC_LAB.ORDERS
  GROUP BY ORDER_TS::DATE;

-- Grant immediately — consumer access is part of the deployment
GRANT SELECT ON TABLE {{my_db}}.RBAC_LAB.ORDER_SUMMARY TO ROLE {{my_analyst_role}};

-- Verify: {{my_analyst_role}} can query it immediately
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.ORDER_SUMMARY LIMIT 3;


> :white_check_mark: **No delay, no ticket, no waiting.** The analyst can query the table the moment it's created because the GRANT shipped alongside the CREATE. This is the data engineer's responsibility.

---

### Optional: Automate with a stored procedure

If you want to see how this scales beyond individual statements, here's a [stored procedure](https://docs.snowflake.com/en/developer-guide/stored-procedure/stored-procedures-overview) that wraps the "create + grant" pattern into a reusable deployment helper. **This is optional** — run it if you're curious about how teams automate access at scale.

The procedure creates any table from a query and immediately grants SELECT to a specified role — ensuring no table is ever deployed without access.

> :point_right: **Note:** This creates a **different** table (`REVENUE_BY_DATE`) from the one you built manually above (`ORDER_SUMMARY`). The point is to demonstrate that the procedure automates the same CREATE + GRANT pattern — you just pass parameters instead of writing the statements by hand. Same concept, reusable tool.

In [ ]:
-- OPTIONAL: A reusable deployment helper that ensures every table ships with its grant
USE ROLE {{my_role}};

CREATE OR REPLACE PROCEDURE {{my_db}}.RBAC_LAB.DEPLOY_TABLE_WITH_ACCESS(
  TABLE_NAME STRING,
  SOURCE_QUERY STRING,
  CONSUMER_ROLE STRING
)
RETURNS STRING
LANGUAGE SQL
AS
BEGIN
  -- Step 1: Create the table
  EXECUTE IMMEDIATE 'CREATE OR REPLACE TABLE ' || :TABLE_NAME || ' AS ' || :SOURCE_QUERY;
  
  -- Step 2: Grant SELECT to the consumer role
  EXECUTE IMMEDIATE 'GRANT SELECT ON TABLE ' || :TABLE_NAME || ' TO ROLE ' || :CONSUMER_ROLE;
  
  RETURN 'Deployed ' || :TABLE_NAME || ' with SELECT granted to ' || :CONSUMER_ROLE;
END;

-- Test it: deploy a new table with automatic access
CALL {{my_db}}.RBAC_LAB.DEPLOY_TABLE_WITH_ACCESS(
  '{{my_db}}.RBAC_LAB.REVENUE_BY_DATE',
  'SELECT ORDER_TS::DATE AS order_date, SUM(TOTAL_AMOUNT) AS daily_revenue FROM {{my_db}}.RBAC_LAB.ORDERS GROUP BY ORDER_TS::DATE',
  '{{my_analyst_role}}'
);


In [ ]:
-- Verify the procedure worked: {{my_analyst_role}} can query the new table immediately
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.REVENUE_BY_DATE LIMIT 3;


> :bulb: **Why this is a Data Engineer pattern:** Analysts consume data — they don't deploy it. Admins govern access policies — they don't build pipelines. The data engineer sits in the middle: you CREATE the objects AND you GRANT the access as part of your delivery workflow. This procedure codifies that responsibility into a reusable tool.
>
> In production, this pattern might live in a dbt post-hook, a CI/CD pipeline step, or a Snowflake Task that runs after each pipeline load.

---
## Clean up — restore to true original state

This lab is one of three RBAC persona labs (Data Analyst, Data Engineer, Platform Admin). They share the same database and objects. The cleanup below restores the environment to the **true original state** from Section 2:
- `{{my_analyst_role}}` has ORDERS access but **NOT** FEEDBACK (the deliberate gap)
- Future grants are revoked (so other labs can demonstrate their own grant scenarios)
- Lab-specific tables are dropped (DAILY_SALES, ORDER_SUMMARY, REVENUE_BY_DATE)
- The deployment procedure is removed
- All base tables (ORDERS, FEEDBACK) and schemas remain intact

This ensures other persona labs start from the expected baseline.


In [ ]:
-- Clean up Step 1: Remove future grants (may already be revoked — that's OK)
USE ROLE {{my_role}};
CALL PLATPLUS_DB.PUBLIC.REVOKE_FUTURE_GRANTS('{{my_db}}.RBAC_LAB', '{{my_analyst_role}}');


In [ ]:
-- Clean up Step 2: Restore baseline grants and drop lab artifacts
-- (This cell always runs regardless of whether Step 1 succeeded)
USE ROLE {{my_role}};

-- Explicitly revoke FEEDBACK (safety net — ensures the deliberate gap is restored)
REVOKE SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK FROM ROLE {{my_analyst_role}};

-- Restore the expected baseline: warehouse + database + schema + ORDERS only
GRANT USAGE  ON DATABASE {{my_db}}                    TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB           TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS    TO ROLE {{my_analyst_role}};

-- Drop lab-specific artifacts from Sections 5 and 7
DROP TABLE IF EXISTS {{my_db}}.RBAC_LAB.DAILY_SALES;
DROP TABLE IF EXISTS {{my_db}}.RBAC_LAB.ORDER_SUMMARY;
DROP TABLE IF EXISTS {{my_db}}.RBAC_LAB.REVENUE_BY_DATE;
DROP PROCEDURE IF EXISTS {{my_db}}.RBAC_LAB.DEPLOY_TABLE_WITH_ACCESS(STRING, STRING, STRING);


## Verify — Original state restored

The cell below confirms {{my_analyst_role}} can access ORDERS (the granted table) but the second cell will fail on FEEDBACK (the deliberate gap is back).

> :stop_sign: **The second verification cell will intentionally fail** — this proves the cleanup worked.


In [ ]:
-- Verify Part 1: ORDERS is accessible (should succeed)
USE ROLE {{my_analyst_role}};
SELECT COUNT(*) AS orders_count FROM {{my_db}}.RBAC_LAB.ORDERS;


In [ ]:
-- Verify Part 2: FEEDBACK is NOT accessible (should fail — proves original state restored)
SELECT COUNT(*) AS feedback_count FROM {{my_db}}.RBAC_LAB.FEEDBACK;


---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Privileges go to **roles**, roles go to users — that's RBAC
- The full grant chain: **Warehouse → Database → Schema → Table** — missing any link = access denied
- "Does not exist or not authorized" does NOT necessarily mean the object is missing — it may exist, but your role can't see it
- An object **owner** can grant and revoke privileges on what they own
- Use `GRANT SELECT ON FUTURE TABLES` to automate access for new objects
- Combine `ON ALL TABLES` (existing) + `ON FUTURE TABLES` (new) for complete coverage
- Use schema-wide grants for analytics schemas; per-object grants for sensitive data
- **Never deploy a table without its grants** — CREATE + GRANT ship together as one atomic unit
- Stored procedures can codify deployment patterns into reusable tools
- **CoCo is your access automation partner** — ask it to generate grant setups, future grants, or diagnose failures

**The Other Personas:**
- The **Data Analyst** notebook covers introspecting access and secondary roles
- The **Platform Admin** notebook covers auditing, revoking, and governance

> :bulb: Each persona lab is independent — you can run them in any order. The cleanup cell restores the environment to baseline between labs.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.